# 🤖 Step 4 — Nutrition Q&A Chatbot + Gradio Demo App
### Multimodal Food Recognition & Nutrition Assistant

**Goal:** Combine all previous steps into one interactive app.

**Full Pipeline:**
```
User uploads photo
       ↓
CLIP (Step 2) → identifies food
       ↓
USDA Lookup (Step 1) → gets nutrition data
       ↓
FAISS RAG (Step 3) → retrieves relevant recipes
       ↓
LLM (Step 4) → answers any nutrition question
       ↓
Gradio UI → beautiful interactive demo
```

## ⚙️ 4.1 — Install Dependencies

In [ ]:
!pip install -q transformers torch torchvision sentence-transformers faiss-cpu gradio Pillow requests accelerate bitsandbytes
print('✅ Dependencies installed')

## 📦 4.2 — Import Libraries & Mount Drive

In [ ]:
import os, pickle, warnings, urllib.request
import numpy as np
import pandas as pd
import torch
import faiss
import gradio as gr
from PIL import Image
from io import BytesIO
from transformers import (
    CLIPModel, CLIPProcessor,
    AutoTokenizer, AutoModelForCausalLM, pipeline
)
from sentence_transformers import SentenceTransformer
from google.colab import drive
warnings.filterwarnings('ignore')

drive.mount('/content/drive')

# ─── CONFIGURE YOUR PATH ─────────────────────────────────────────────────
BASE       = '/content/drive/MyDrive/'
OUTPUT_DIR = BASE + 'nutrition_assistant_outputs/'
# ─────────────────────────────────────────────────────────────────────────

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ Libraries ready | Device: {DEVICE}')

## 📥 4.3 — Load All Assets from Previous Steps

In [ ]:
print('Loading all assets...\n')

# ── Step 1: Master nutrition table ───────────────────────────────────────
nutrition_wide = pd.read_csv(OUTPUT_DIR + 'master_nutrition.csv')
print(f'✅ Nutrition table  : {len(nutrition_wide):,} foods')

# ── Step 2: CLIP model ───────────────────────────────────────────────────
clip_model     = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(DEVICE)
clip_processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
clip_model.eval()
print(f'✅ CLIP model       : loaded')

# ── Step 3: FAISS index + recipes ────────────────────────────────────────
faiss_index    = faiss.read_index(OUTPUT_DIR + 'recipe_faiss.index')
embed_model    = SentenceTransformer('all-MiniLM-L6-v2')
subset_path    = OUTPUT_DIR + 'recipes_subset.csv'
recipes_df     = pd.read_csv(subset_path if os.path.exists(subset_path)
                             else OUTPUT_DIR + 'recipes_clean.csv')
print(f'✅ FAISS index      : {faiss_index.ntotal:,} recipes')
print(f'✅ Recipes loaded   : {len(recipes_df):,} recipes')

## 🏷️ 4.4 — Rebuild CLIP Food Vocabulary & Text Embeddings

In [ ]:
FOOD_LABELS = [
    'pizza', 'burger', 'hamburger', 'sandwich', 'hot dog', 'sushi',
    'fried chicken', 'grilled chicken', 'chicken curry', 'butter chicken',
    'biryani', 'fried rice', 'noodles', 'pasta', 'spaghetti', 'lasagna',
    'tacos', 'burrito', 'quesadilla', 'nachos', 'fish and chips',
    'steak', 'grilled fish', 'shrimp', 'soup', 'stew', 'omelette',
    'pancakes', 'waffles', 'french toast', 'scrambled eggs', 'boiled eggs',
    'idli', 'dosa', 'samosa', 'dal', 'palak paneer', 'paneer tikka',
    'roti', 'naan', 'paratha', 'upma', 'poha',
    'french fries', 'onion rings', 'popcorn', 'chips', 'spring rolls',
    'dim sum', 'dumplings', 'kebab', 'cookies', 'chocolate chip cookies',
    'biscuits', 'brownie',
    'apple', 'banana', 'orange', 'mango', 'grapes', 'strawberry',
    'watermelon', 'pineapple', 'avocado', 'blueberries', 'peach',
    'salad', 'broccoli', 'carrot', 'tomato', 'cucumber', 'corn',
    'spinach', 'mushroom', 'bell pepper', 'cauliflower', 'potato',
    'ice cream', 'cake', 'cupcake', 'donut', 'croissant', 'muffin',
    'bread', 'toast', 'cheese', 'yogurt', 'butter',
    'coffee', 'tea', 'juice', 'smoothie', 'milkshake',
    'rice', 'oatmeal', 'cereal', 'lentils', 'chickpeas', 'beans'
]

LABEL_PROMPTS = [f'a photo of {label}' for label in FOOD_LABELS]

with torch.no_grad():
    text_inputs = clip_processor(text=LABEL_PROMPTS, return_tensors='pt',
                                 padding=True, truncation=True).to(DEVICE)
    raw_out     = clip_model.text_model(**text_inputs)
    text_embeds = clip_model.text_projection(raw_out.pooler_output)
    text_embeds = text_embeds / text_embeds.norm(p=2, dim=-1, keepdim=True)

print(f'✅ CLIP vocabulary  : {len(FOOD_LABELS)} food labels embedded')

## 🤖 4.5 — Load LLM (Flan-T5 — No API Key Needed)

In [ ]:
# Using google/flan-t5-large — free, no API key, runs on Colab GPU
# Excellent at instruction-following and Q&A tasks
print('Loading LLM (Flan-T5-Large)... (~800MB, cached after first run)')

LLM_NAME  = 'google/flan-t5-large'
tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
llm       = AutoModelForCausalLM.from_pretrained(
    LLM_NAME,
    device_map='auto',
    torch_dtype=torch.float16 if DEVICE == 'cuda' else torch.float32
)

llm_pipeline = pipeline(
    'text2text-generation',
    model=LLM_NAME,
    tokenizer=tokenizer,
    max_new_tokens=300,
    device=0 if DEVICE == 'cuda' else -1
)

print('✅ LLM ready')

## 🔧 4.6 — Build All Pipeline Functions

In [ ]:
# ── CLIP: Image → Food Name ───────────────────────────────────────────────
def predict_food(image: Image.Image, top_n: int = 5):
    with torch.no_grad():
        inputs       = clip_processor(images=image, return_tensors='pt').to(DEVICE)
        raw_out      = clip_model.vision_model(**inputs)
        image_embeds = clip_model.visual_projection(raw_out.pooler_output)
        image_embeds = image_embeds / image_embeds.norm(p=2, dim=-1, keepdim=True)
        similarity   = (image_embeds @ text_embeds.T).squeeze(0) * 100.0
        probs        = similarity.softmax(dim=-1).cpu().numpy()
    top_idx = probs.argsort()[::-1][:top_n]
    return [{'food': FOOD_LABELS[i], 'confidence': float(probs[i])} for i in top_idx]


# ── USDA: Food Name → Nutrition Summary ──────────────────────────────────
def get_nutrition_summary(food_query: str) -> str:
    query    = food_query.lower().strip()
    keywords = query.split()
    mask     = nutrition_wide['food_name_clean'].str.contains(keywords[0], na=False)
    for kw in keywords[1:]:
        mask &= nutrition_wide['food_name_clean'].str.contains(kw, na=False)
    results = nutrition_wide[mask]
    if results.empty:
        mask    = nutrition_wide['food_name_clean'].str.contains(keywords[0], na=False)
        results = nutrition_wide[mask]
    if results.empty:
        return f'No USDA nutrition data found for "{food_query}".'
    row = results.iloc[0]
    def v(col):
        val = row.get(col)
        return f'{val:.1f}' if pd.notna(val) else 'N/A'
    return f"""NUTRITION (per 100g): {row['food_name']}
Category: {row.get('category_name', 'Unknown')}
Calories: {v('calories_kcal')} kcal | Protein: {v('protein_g')}g | Fat: {v('fat_g')}g
Carbs: {v('carbs_g')}g | Fiber: {v('fiber_g')}g | Sugar: {v('sugar_g')}g
Sodium: {v('sodium_mg')}mg | Cholesterol: {v('cholesterol_mg')}mg
Calcium: {v('calcium_mg')}mg | Iron: {v('iron_mg')}mg
Vitamin C: {v('vitamin_c_mg')}mg | Vitamin A: {v('vitamin_a_mcg')}mcg"""


# ── FAISS: Food Name → Matching Recipes ──────────────────────────────────
def get_recipe_context(food_name: str, top_k: int = 3) -> str:
    query_embed = embed_model.encode([food_name], convert_to_numpy=True)
    faiss.normalize_L2(query_embed)
    scores, indices = faiss_index.search(query_embed, top_k)
    results = recipes_df.iloc[indices[0]].copy()
    results['score'] = scores[0]
    context = f'RELEVANT RECIPES FOR "{food_name}":\n'
    for i, row in results.iterrows():
        context += f'- {row["title"]}: {str(row["ingredients_text"])[:150]}\n'
    return context


# ── LLM: Answer Any Nutrition Question ───────────────────────────────────
def ask_llm(question: str, food_name: str, nutrition_ctx: str, recipe_ctx: str) -> str:
    prompt = f"""You are a helpful nutrition assistant. Use the context below to answer the question.

IDENTIFIED FOOD: {food_name}

{nutrition_ctx}

{recipe_ctx}

USER QUESTION: {question}

Give a clear, helpful, and specific answer based on the nutrition data and recipes above.
Answer:"""
    response = llm_pipeline(prompt, max_new_tokens=300, do_sample=False)
    return response[0]['generated_text'].strip()


print('✅ All pipeline functions ready')

## 🎨 4.7 — Build the Gradio App

In [ ]:
# ── Stateful context store (persists between chatbot turns) ───────────────
app_state = {'food_name': None, 'nutrition': None, 'recipes': None}


def analyze_image(image):
    """
    Called when user uploads an image.
    Returns: food label, confidence, nutrition panel, recipe panel.
    """
    if image is None:
        return 'No image uploaded.', '', '', []

    pil_image    = Image.fromarray(image).convert('RGB')
    predictions  = predict_food(pil_image, top_n=5)
    food_name    = predictions[0]['food']
    confidence   = predictions[0]['confidence'] * 100

    nutrition    = get_nutrition_summary(food_name)
    recipes      = get_recipe_context(food_name, top_k=3)

    # Store in state for chatbot
    app_state['food_name'] = food_name
    app_state['nutrition'] = nutrition
    app_state['recipes']   = recipes

    # Build top-5 predictions table
    pred_table = [[p['food'].title(), f"{p['confidence']*100:.1f}%"]
                  for p in predictions]

    header  = f'## 🍽️ Identified: **{food_name.title()}** ({confidence:.1f}% confidence)'
    return header, nutrition, recipes, pred_table


def chat(user_message, history):
    """
    Called on each chatbot message.
    Uses stored food context to answer questions.
    """
    if app_state['food_name'] is None:
        response = 'Please upload a food image first so I can identify it and answer your questions!'
    else:
        response = ask_llm(
            question    = user_message,
            food_name   = app_state['food_name'],
            nutrition_ctx = app_state['nutrition'],
            recipe_ctx  = app_state['recipes']
        )
    history.append((user_message, response))
    return history, ''


# ── Build the Gradio Interface ─────────────────────────────────────────────
with gr.Blocks(
    theme=gr.themes.Soft(),
    title='Food Vision Nutrition Assistant'
) as demo:

    gr.Markdown("""
    # 🍽️ Multimodal Food Recognition & Nutrition Assistant
    **Upload a food photo → Get instant nutrition info + recipes + AI Q&A**
    """)

    with gr.Row():
        # Left column: image upload
        with gr.Column(scale=1):
            image_input  = gr.Image(label='📸 Upload Food Photo', type='numpy')
            analyze_btn  = gr.Button('🔍 Analyze Food', variant='primary', size='lg')
            food_header  = gr.Markdown('')
            pred_table   = gr.Dataframe(
                headers=['Food', 'Confidence'],
                label='Top 5 CLIP Predictions',
                interactive=False
            )

        # Right column: nutrition + recipes
        with gr.Column(scale=1):
            with gr.Tab('🥗 Nutrition Info'):
                nutrition_box = gr.Textbox(
                    label='USDA Nutrition Data (per 100g)',
                    lines=14, interactive=False
                )
            with gr.Tab('📖 Matching Recipes'):
                recipes_box = gr.Textbox(
                    label='Top Matching Recipes from Dataset',
                    lines=14, interactive=False
                )

    gr.Markdown('---')
    gr.Markdown('## 💬 Ask the Nutrition Assistant')

    chatbot      = gr.Chatbot(label='Nutrition Q&A', height=350)
    with gr.Row():
        chat_input = gr.Textbox(
            placeholder='Ask anything: "Is this healthy?", "How many calories if I eat 200g?", "Give me a healthier recipe"...',
            label='Your Question',
            scale=5
        )
        chat_btn   = gr.Button('Send', variant='primary', scale=1)

    gr.Examples(
        examples=[
            ['Is this food healthy for weight loss?'],
            ['How many calories if I eat 250g of this?'],
            ['Is this safe for a diabetic person?'],
            ['What nutrients am I missing if I eat only this?'],
            ['Give me a healthier recipe alternative.'],
        ],
        inputs=chat_input
    )

    # ── Wire up events ──────────────────────────────────────────────────
    analyze_btn.click(
        fn      = analyze_image,
        inputs  = [image_input],
        outputs = [food_header, nutrition_box, recipes_box, pred_table]
    )
    chat_btn.click(
        fn      = chat,
        inputs  = [chat_input, chatbot],
        outputs = [chatbot, chat_input]
    )
    chat_input.submit(
        fn      = chat,
        inputs  = [chat_input, chatbot],
        outputs = [chatbot, chat_input]
    )

print('✅ Gradio app built successfully')

## 🚀 4.8 — Launch the App!

In [ ]:
# share=True gives you a public URL valid for 72 hours
# Perfect for demos, presentations, and award submissions
demo.launch(share=True, debug=False)

## ✅ Full Project Complete!

| Step | Component | Status |
|---|---|---|
| Step 1 | Data Pipeline (USDA merge) | ✅ |
| Step 2 | CLIP Food Image Recognition | ✅ |
| Step 3 | FAISS Recipe RAG Engine | ✅ |
| Step 4 | LLM Chatbot + Gradio App | ✅ |

**Your app can:**
- 📸 Identify food from any photo
- 🥗 Show full USDA nutrition breakdown
- 📖 Match relevant recipes from 2.2M recipe dataset
- 💬 Answer any nutrition question about the identified food
- 🌐 Share a public demo URL instantly

**Share your demo link for award submissions!** 🏆